In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools

additional_packages = [
    'org.apache.sedona:sedona-spark-3.5_2.12:1.7.1',
    'org.datasyslab:geotools-wrapper:1.7.1-28.5',
]

config = SedonaContext.builder().\
     config("spark.jars.packages", ",".join(additional_packages)).\
     config("spark.sql.autoBroadcastJoinThreshold", "-1").\
     config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
     config("spark.hadoop.fs.s3a.access.key", "sedona"). \
     config("spark.hadoop.fs.s3a.secret.key", "sedona_password"). \
     config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000"). \
     config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem").\
     config("spark.hadoop.fs.s3a.path.style.access", "true"). \
     getOrCreate()

sedona = SedonaContext.create(config)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4953b69d-9679-40d3-8608-7948687b3076;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.7.1 in central
	found org.apache.sedona#sedona-common;1.7.1 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-

In [3]:
(sedona
    .read
    .format("binaryFile")
    .load("s3a://sedona/source_data/fdi_data")
    .selectExpr("RS_FromGeoTiff(content) AS rast")
    .createOrReplaceTempView("ffdi"))

(
    sedona
        .read
        .format("binaryFile")
        .load("s3a://sedona/source_data/world_population_raster")
        .selectExpr("RS_FromGeoTiff(content) AS rast")
        .createOrReplaceTempView("population")
)

In [4]:
sedona.sql(
    """
    SELECT 
        raster.tile as rast,
        raster.x,
        raster.y 
    FROM ffdi
    LATERAL VIEW RS_TileExplode(rast, 100, 100) raster
    """
).createOrReplaceTempView("fdi_tiles")

In [5]:
geometry = sedona.sql(
    """
    WITH pixelized AS (
        SELECT 
            RS_PixelAsPolygons(rast, 1) AS pixels,
            x,
            y
        FROM fdi_tiles
    ),
    classified AS (
        SELECT
            pixel.geom,
            x,
            y,
            CASE
                WHEN pixel.value > 50 THEN 'extreme'
                WHEN pixel.value > 25 THEN 'very high'
                WHEN pixel.value > 12 THEN 'high'
                WHEN pixel.value > 5 THEN 'moderate'
                WHEN pixel.value > 0 THEN 'low'
            END AS fire_danger_class
        FROM pixelized
        LATERAL VIEW explode(pixels) AS pixel
        WHERE pixel.value > 0 AND pixel.value < 255
    )
     SELECT
            ST_Union_Aggr(geom) AS geom,
            fire_danger_class
        FROM classified
        GROUP BY fire_danger_class, x, y
    """
).createOrReplaceTempView("fire_danger")

In [6]:
sedona.sql("SELECT * FROM fire_danger").show(5)

[Stage 3:>                                                          (0 + 1) / 1]

+--------------------+-----------------+
|                geom|fire_danger_class|
+--------------------+-----------------+
|MULTIPOLYGON (((-...|             high|
|MULTIPOLYGON (((7...|              low|
|MULTIPOLYGON (((3...|         moderate|
|MULTIPOLYGON (((7...|         moderate|
|MULTIPOLYGON (((-...|             high|
+--------------------+-----------------+
only showing top 5 rows



In [8]:
## SELECT  RS_ZonalStats(rast, 1, ST_CollectionExtrac(geom), 1, 'sum', true, false) AS population_sum
sedona.sql(
    
    """
    WITH intersection AS (
        SELECT 
            rast,
            ST_Buffer(ST_Intersection(RS_Envelope(rast), geom), -0.0001) AS geom,
            fire_danger_class
        FROM population AS p
        JOIN fire_danger AS f ON RS_Intersects(p.rast, f.geom)
    ),
    zonal_stats AS (
        SELECT 
            RS_ZonalStats(rast, geom, 1, 'sum') AS population_sum,
            fire_danger_class
        FROM intersection
    )
    SELECT 
        fire_danger_class,
        CAST(sum(population_sum) AS DECIMAL(38, 0)) AS population_sum
    FROM zonal_stats
    GROUP BY fire_danger_class

    """
).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[fire_danger_class#77], functions=[sum(population_sum#138)])
   +- Exchange hashpartitioning(fire_danger_class#77, 200), ENSURE_REQUIREMENTS, [plan_id=278]
      +- HashAggregate(keys=[fire_danger_class#77], functions=[partial_sum(population_sum#138)])
         +- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_ZonalStats**   AS population_sum#138, fire_danger_class#77]
            +- RangeJoin rast#65: raster, geom#75: geometry, INTERSECTS,  **org.apache.spark.sql.sedona_sql.expressions.raster.RS_Intersects**
               :- Project [ **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**   AS rast#65]
               :  +- Filter isnotnull( **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTiff**  )
               :     +- FileScan binaryFile [content#60] Batched: false, DataFilters: [isnotnull( **org.apache.spark.sql.sedona_sql.expressions.raster.RS_FromGeoTif